# 04 · TF-IDF + Ridge hyperparameter sweeps (Days 8-10)

Three sequential development-set sweeps to fix the baseline TF-IDF + Ridge
configuration for Empathy prediction. Each sweep holds the other
hyperparameters fixed so its effect can be isolated.

1. **Day 8 — n-gram range.** Unigrams vs unigrams+bigrams.
2. **Day 9 — vocabulary size.** `max_features` ∈ {1000, 5000, 10000, 20000}.
3. **Day 10 — regularization strength.** Ridge α ∈ {0.1, 1, 3, 10, 100}.

**Final configuration** (justified by the sweeps below):
`TfidfVectorizer(ngram_range=(1,1), max_features=10000)` with
`Ridge(alpha=3.0)`, trained on the training split and evaluated on both dev
and test.

**Split note.** This notebook uses the same internal conversation-grouped
70/15/15 split as `02_baseline_multi_target.ipynb`, kept for consistency
with the exploratory experiments in that notebook. All hyperparameter
decisions are made on dev; test is used only for final reporting.


## 0 · Setup


In [6]:
!git clone https://github.com/DavorSopar/thesis-empathy.git /content/thesis-empathy
import sys
sys.path.insert(0, '/content/thesis-empathy/src')

fatal: destination path '/content/thesis-empathy' already exists and is not an empty directory.


In [16]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
# Make src/ importable whether run from notebooks/ or the repo root.
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "src").is_dir():
    pass
elif (REPO_ROOT.parent / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
from data import load_convt, impute_selfdisclosure, TARGETS

RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42

## 1 · Load the three official WASSA splits

The WASSA-released training file contains all conversations from the
underlying corpus, including those subsequently released as development
and test splits. We filter the training set to exclude any conversation
appearing in dev or test, yielding a conversation-disjoint partition.


In [17]:
# Load all three splits
train_raw = load_convt('train')
dev = load_convt('dev')
test = load_convt('test')

# Filter train: remove any conversation appearing in dev or test
excluded_ids = set(dev.conversation_id) | set(test.conversation_id)
train = train_raw[~train_raw.conversation_id.isin(excluded_ids)].reset_index(drop=True)

# Sanity check: no conversation overlap between splits
assert not (set(train.conversation_id) & set(dev.conversation_id))
assert not (set(train.conversation_id) & set(test.conversation_id))
assert not (set(dev.conversation_id) & set(test.conversation_id))

split_tbl = pd.DataFrame({
    'turns':         [len(train), len(dev), len(test)],
    'conversations': [train.conversation_id.nunique(),
                      dev.conversation_id.nunique(),
                      test.conversation_id.nunique()],
}, index=['train', 'dev', 'test'])
print(split_tbl)
print('\nAll three splits are conversation-disjoint.')


       turns  conversations
train   9330            405
dev      990             33
test    2061             63

All three splits are conversation-disjoint.


## 2 · Metric helpers


In [18]:
def pearson(y_true, y_pred):
    """Pearson r; returns NaN when either side is constant (undefined)."""
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    if np.std(y_true) < 1e-12 or np.std(y_pred) < 1e-12:
        return np.nan
    return float(np.corrcoef(y_true, y_pred)[0, 1])

def score_all(y_true, y_pred):
    return {
        "pearson": pearson(y_true, y_pred),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }

## 1 · Day 8 — n-gram range

**Goal.** Test whether phrase-level features (bigrams) capture empathy
signal beyond individual words. Unigrams alone treat `"not sorry"` as
the same as `"sorry not"`; adding bigrams preserves short phrases that
encode negation, intensity, and idiom.

**Setup.** Two `TfidfVectorizer` configurations, everything else fixed
(`max_features=5000`, `Ridge(alpha=3.0)`, dev evaluation).


In [19]:
BEST_ALPHA = 3.0
TARGET = 'Empathy'

configs = [
    {"name": "unigrams",         "ngram_range": (1, 1)},
    {"name": "unigrams+bigrams", "ngram_range": (1, 2)},
]

y_train = train[TARGET]
y_dev   = dev[TARGET]

ngram_results = []
for cfg in configs:
    vectorizer = TfidfVectorizer(ngram_range=cfg["ngram_range"], max_features=5000)
    X_train = vectorizer.fit_transform(train["text"])
    X_dev   = vectorizer.transform(dev["text"])

    model = Ridge(alpha=BEST_ALPHA).fit(X_train, y_train)
    preds = model.predict(X_dev)

    mae  = mean_absolute_error(y_dev, preds)
    rmse = np.sqrt(mean_squared_error(y_dev, preds))
    r    = pearsonr(y_dev, preds)[0]

    ngram_results.append({
        "name": cfg["name"],
        "ngram_range": str(cfg["ngram_range"]),
        "mae": mae,
        "rmse": rmse,
        "pearson": r,
    })
    print(f"{cfg['name']:20s} MAE={mae:.4f} RMSE={rmse:.4f} Pearson={r:.4f}")

# Save
pd.DataFrame(ngram_results).to_csv(RESULTS_DIR / 'day08_ngram_experiment.csv', index=False)
print("\nSaved results/day08_ngram_experiment.csv")

unigrams             MAE=0.7661 RMSE=0.9252 Pearson=0.5462
unigrams+bigrams     MAE=0.7640 RMSE=0.9253 Pearson=0.5471

Saved results/day08_ngram_experiment.csv


**Result.** Unigrams and unigrams+bigrams produced essentially identical
dev performance (Pearson 0.5462 vs 0.5471) — a difference well within
noise. Phrase-level features do not add meaningful signal for empathy
prediction on this data at this feature dimensionality. Unigrams retained
for baseline simplicity.


## 2 · Day 9 — vocabulary size (`max_features`)

**Goal.** Establish the vocabulary size where performance saturates. Too
few features drops informative words (underfitting); too many either adds
rare noisy features or is bounded by the corpus's actual vocabulary size.

**Setup.** Winning ngram range from Day 8 (`(1,1)`), fixed `alpha=3.0`,
sweep `max_features ∈ {1000, 5000, 10000, 20000}` on dev.


In [21]:
# Using the winning ngram_range from Day 8, train Ridge with max_features = 1000, 5000, 10000, 20000.
configs1 = [1000, 5000, 10000, 20000]

maxfeat_results = []
for cfg in configs1:
  vec1 = TfidfVectorizer(ngram_range=(1,1), max_features=cfg)
  X_tr = vec1.fit_transform(train["text"])
  X_dv = vec1.transform(dev["text"])
  model = Ridge(alpha=3.0).fit(X_tr, y_train)
  preds = model.predict(X_dv)
  mae = mean_absolute_error(y_dev, preds)
  rmse = np.sqrt(mean_squared_error(y_dev, preds))
  r = pearsonr(y_dev, preds)[0]
  maxfeat_results.append({"size": cfg, "mae": mae, "rmse": rmse, "pearson": r})
  print(f"Vocabulary size={cfg} MAE={mae:.4f} RMSE={rmse:.4f} Pearson={r:.4f}")
pd.DataFrame(maxfeat_results).to_csv(RESULTS_DIR / 'day09_maxfeatures_sweep.csv', index=False)

Vocabulary size=1000 MAE=0.7656 RMSE=0.9278 Pearson=0.5398
Vocabulary size=5000 MAE=0.7661 RMSE=0.9252 Pearson=0.5462
Vocabulary size=10000 MAE=0.7663 RMSE=0.9259 Pearson=0.5457
Vocabulary size=20000 MAE=0.7663 RMSE=0.9259 Pearson=0.5457


In [22]:
print(f"Vocabulary size requested: {cfg}, actual: {len(vec1.get_feature_names_out())}")

Vocabulary size requested: 20000, actual: 8856


The `max_features` sweep across {1000, 5000, 10000, 20000} shows essentially
flat performance (dev Pearson r = 0.540 to 0.546), with 10,000 and 20,000
producing identical results because they both use the full training
vocabulary (~7,X00 unigrams). We select max_features = 5000, which achieves
the highest dev Pearson (0.5462) while being the simplest configuration.

The flat curve indicates that on the WASSA CONV-Turn task, vocabulary size
is not the limiting factor for lexical-feature models — additional features
add no meaningful signal. This is consistent with the broader finding
(Section X, error analysis) that turn-level empathy prediction is bottlenecked
by the pragmatic-context gap between what turn text carries and what
conversation-context annotations encode, rather than by feature dimensionality.

## 3 · Day 10 — regularization strength (α)

**Goal.** Confirm the optimal Ridge regularization strength for the final
configuration. Small α weakens the L2 penalty and lets coefficients grow
large, allowing the model to fit training-specific noise (overfitting).
Large α forces coefficients toward zero regardless of the data, preventing
the model from capturing real patterns (underfitting).

**Setup.** Winning ngram and vocabulary size from Days 8-9, sweep
`α ∈ {0.1, 1, 3, 10, 100}` on dev.


In [23]:
alphas = [0.1, 1.0, 3.0, 10.0, 100.0]

vec = TfidfVectorizer(ngram_range=(1,1), max_features=10000)
X_tr = vec.fit_transform(train["text"])
X_dv = vec.transform(dev["text"])

results = []
for alpha in alphas:
    model = Ridge(alpha=alpha).fit(X_tr, y_train)
    preds = model.predict(X_dv)
    mae = mean_absolute_error(y_dev, preds)
    rmse = np.sqrt(mean_squared_error(y_dev, preds))
    r = pearsonr(y_dev, preds)[0]
    results.append({"alpha": alpha, "mae": mae, "rmse": rmse, "pearson": r})
    print(f"alpha={alpha:>6}: MAE={mae:.4f} RMSE={rmse:.4f} Pearson={r:.4f}")

pd.DataFrame(results).to_csv(RESULTS_DIR / 'day10_alpha_sweep.csv', index=False)
print("Saved results/day10_alpha_sweep.csv")

alpha=   0.1: MAE=0.7984 RMSE=0.9836 Pearson=0.4797
alpha=   1.0: MAE=0.7632 RMSE=0.9262 Pearson=0.5377
alpha=   3.0: MAE=0.7663 RMSE=0.9259 Pearson=0.5457
alpha=  10.0: MAE=0.7848 RMSE=0.9427 Pearson=0.5404
alpha= 100.0: MAE=0.8503 RMSE=1.0179 Pearson=0.5030
Saved results/day10_alpha_sweep.csv


**Result.** Dev Pearson traced a clean U-shaped bias-variance curve:
0.4797 at α=0.1, rising to a peak of 0.5457 at α=3.0, then falling to
0.5030 at α=100. Both extremes clearly underperform, confirming the
expected bias-variance tradeoff. **α=3.0 selected as final regularization
strength.**


## 4 · Final Ridge baseline

Retrain on the training split with the fully-justified configuration and
evaluate on both dev and test. These are the canonical Ridge baseline
numbers cited throughout the thesis.

**Configuration:** `TfidfVectorizer(ngram_range=(1,1), max_features=10000)`
with `Ridge(alpha=3.0)`, target = Empathy.


In [24]:
# Final config from Days 8-10 sweeps
BEST_ALPHA = 3.0

# Rebuild vectorizer and transform all three splits
vec = TfidfVectorizer(ngram_range=(1,1), max_features=10000)
X_tr = vec.fit_transform(train["text"])
X_dv = vec.transform(dev["text"])
X_te = vec.transform(test["text"])

y_train = train["Empathy"]
y_dev = dev["Empathy"]
y_test = test["Empathy"]

# Train the final Ridge model
final_ridge = Ridge(alpha=BEST_ALPHA).fit(X_tr, y_train)

# Evaluation helper
def evaluate(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r = pearsonr(y_true, y_pred)[0]
    print(f"{label:35s}  MAE={mae:.4f}  RMSE={rmse:.4f}  Pearson={r:.4f}")
    return {"label": label, "mae": mae, "rmse": rmse, "pearson": r}

# Evaluate on both splits
dev_results = evaluate(y_dev, final_ridge.predict(X_dv), "Final Ridge baseline · dev")
test_results = evaluate(y_test, final_ridge.predict(X_te), "Final Ridge baseline · test")

Final Ridge baseline · dev           MAE=0.7663  RMSE=0.9259  Pearson=0.5457
Final Ridge baseline · test          MAE=0.9728  RMSE=1.1740  Pearson=0.4926


### Persist the final model and result table

Saved to `results/` and `models/` so downstream notebooks and the thesis
write-up can reference these numbers and reload the model without
retraining.


In [25]:
import joblib

MODELS_DIR = REPO_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

# Save the metrics table
pd.DataFrame([dev_results, test_results]).to_csv(
    RESULTS_DIR / 'final_ridge_baseline_results.csv', index=False
)

# Save the trained model and its vectorizer
joblib.dump(final_ridge, MODELS_DIR / 'final_ridge_baseline.pkl')
joblib.dump(vec,         MODELS_DIR / 'final_ridge_vectorizer.pkl')

print('Saved results/final_ridge_baseline_results.csv')
print('Saved models/final_ridge_baseline.pkl')
print('Saved models/final_ridge_vectorizer.pkl')


Saved results/final_ridge_baseline_results.csv
Saved models/final_ridge_baseline.pkl
Saved models/final_ridge_vectorizer.pkl


## Summary

Three independent development-set sweeps fixed the final Ridge baseline
configuration:

| Sweep | Range tested | Selected |
|---|---|---|
| N-gram range | (1,1), (1,2) | (1,1) |
| max_features | 1000, 5000, 10000, 20000 | 10000 |
| Ridge α | 0.1, 1, 3, 10, 100 | 3.0 |

Final baseline: `TfidfVectorizer(ngram_range=(1,1), max_features=10000)`
with `Ridge(alpha=3.0)`. Numbers reported in `results/final_ridge_baseline_results.csv`
and cited in the thesis Results section.
